# Gradient Scaling Deep Dive

## Overview

This tutorial provides an in-depth look at gradient scaling for mixed precision training.

### Topics Covered
- Why gradient scaling is needed
- Dynamic vs static scaling
- Handling overflow/underflow
- Custom scaling strategies

## 1. The Underflow Problem

### FP16 Numerical Range

```
FP16 Range: [6.1e-5, 65504]

Typical gradient magnitudes:
├── Early layers: 1e-3 to 1e-5 (OK)
├── Deep layers: 1e-6 to 1e-8 (UNDERFLOW!)
└── After many steps: Can get even smaller

Problem: Gradients < 6.1e-5 become ZERO in FP16
Result: Training stalls, model doesn't learn
```

In [ ]:
import torch

# Demonstrate FP16 underflow
def show_fp16_underflow():
    values = [1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8]
    
    print("FP16 Underflow Demonstration:")
    print(f"{'Value':<12} {'FP32':<15} {'FP16':<15} {'Lost?'}")
    print("-" * 50)
    
    for v in values:
        fp32 = torch.tensor(v, dtype=torch.float32)
        fp16 = fp32.half()
        lost = "YES!" if fp16.item() == 0 else "No"
        print(f"{v:<12.0e} {fp32.item():<15.2e} {fp16.item():<15.2e} {lost}")

show_fp16_underflow()

## 2. Loss Scaling Solution

### The Idea

$$\tilde{L} = s \cdot L \quad \Rightarrow \quad \tilde{g} = s \cdot g$$

Scale up loss → gradients scale up → fit in FP16 range → scale back down

In [ ]:
class ManualGradScaler:
    """Simple gradient scaler for understanding the concept."""
    
    def __init__(self, init_scale=65536.0):
        self.scale = init_scale
    
    def scale_loss(self, loss):
        """Scale loss before backward."""
        return loss * self.scale
    
    def unscale_gradients(self, optimizer):
        """Unscale gradients after backward."""
        for group in optimizer.param_groups:
            for param in group['params']:
                if param.grad is not None:
                    param.grad.data /= self.scale
    
    def check_overflow(self, optimizer):
        """Check for inf/nan in gradients."""
        for group in optimizer.param_groups:
            for param in group['params']:
                if param.grad is not None:
                    if torch.isinf(param.grad).any() or torch.isnan(param.grad).any():
                        return True
        return False

## 3. Dynamic Loss Scaling

### Algorithm

```
Initialize: scale = 65536, growth_interval = 2000

Each step:
  1. Scale loss and backward
  2. Check for overflow (inf/nan in gradients)
  3. If overflow:
       - scale = scale / 2
       - Skip optimizer step
       - Reset success counter
  4. If no overflow:
       - Unscale and step optimizer
       - success_count += 1
       - If success_count >= growth_interval:
           scale = scale * 2
           success_count = 0
```

In [ ]:
class DynamicGradScaler:
    """Dynamic gradient scaler with automatic scale adjustment."""
    
    def __init__(self, init_scale=65536.0, growth_factor=2.0,
                 backoff_factor=0.5, growth_interval=2000):
        self.scale = init_scale
        self.growth_factor = growth_factor
        self.backoff_factor = backoff_factor
        self.growth_interval = growth_interval
        self.success_count = 0
        self.overflow_count = 0
    
    def step(self, optimizer, overflow):
        """Update scale based on overflow status."""
        if overflow:
            self.scale *= self.backoff_factor
            self.success_count = 0
            self.overflow_count += 1
            return False  # Skip optimizer step
        else:
            self.success_count += 1
            if self.success_count >= self.growth_interval:
                self.scale *= self.growth_factor
                self.success_count = 0
            return True  # Do optimizer step
    
    def get_stats(self):
        return {
            'scale': self.scale,
            'overflow_count': self.overflow_count,
            'overflow_rate': self.overflow_count / max(1, self.success_count + self.overflow_count)
        }

## 4. Summary

### Best Practices

1. **Start with high scale** (65536) - better to overflow than underflow
2. **Monitor overflow rate** - should be < 1%
3. **Use BF16 if available** - no scaling needed
4. **Check gradient norms** - helps diagnose training issues